In [1]:
import sys
import warnings

from sklearn.utils import resample

warnings.filterwarnings("ignore")
sys.path.append('/home/cdsw/Tony/Mlops_new/Module')
import config
import pandas as pd

[NbConvertApp] WARNING | pattern 'config.ipynb' matched no files
This application is used to convert notebook files (*.ipynb) to various other
formats.


Options
-------

Arguments that take values are actually convenience aliases to full
Configurables, whose aliases are listed on the help line. For more information
on full configurables, see '--help-all'.

--debug
    set log level to logging.DEBUG (maximize logging output)
--generate-config
    generate default config file
-y
    Answer yes to any questions instead of prompting.
--execute
    Execute the notebook prior to export.
--allow-errors
    Continue notebook execution even if one of the cells throws an error and include the error message in the cell output (the default behaviour is to abort conversion). This flag is only relevant if '--execute' was specified, too.
--stdin
    read a single notebook file from stdin. Write the resulting notebook with default basename 'notebook.*'
--stdout
    Write notebook output to stdout ins


Bad key backend.qt4 in file /etc/matplotlib/matplotlibrc, line 43 ('backend.qt4 : PyQt4        # PyQt4 | PySide')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.3.4/matplotlibrc.template
or from the matplotlib source distribution


In [2]:
from Sql_module import get_SQL_raw_data, send_table_to_sql

[NbConvertApp] WARNING | pattern 'Sql_module.ipynb' matched no files
This application is used to convert notebook files (*.ipynb) to various other
formats.


Options
-------

Arguments that take values are actually convenience aliases to full
Configurables, whose aliases are listed on the help line. For more information
on full configurables, see '--help-all'.

--debug
    set log level to logging.DEBUG (maximize logging output)
--generate-config
    generate default config file
-y
    Answer yes to any questions instead of prompting.
--execute
    Execute the notebook prior to export.
--allow-errors
    Continue notebook execution even if one of the cells throws an error and include the error message in the cell output (the default behaviour is to abort conversion). This flag is only relevant if '--execute' was specified, too.
--stdin
    read a single notebook file from stdin. Write the resulting notebook with default basename 'notebook.*'
--stdout
    Write notebook output to stdout

In [3]:
mother_list1= ['潛客', '非潛客']
mother_list2= ['不分潛客']
prod_ym_list_pd_12m = ['流失預警']
do_detail_ym_prod_list = [
 '債券型基金',
 '不限用途',
 '保險商品',
 '台股信用交易',
 '台股定期定額',
 '海外股票定期定額',
 '基金定期定額',
 '儲蓄型保險商品',
 '基金',
 '境內結構型',
 '海外債',
 '平衡型基金',
 '海外股票',
 '結構型商品',
 '股票型基金',
 '境外結構型',
 '投資型保險商品',
 '期貨',
 '雙向借券',
 '財管商品']

In [4]:
def load_product():
    return """
select distinct product from S_IANLEONG."模型代碼對照表" a
where target != '客戶雙週實動意圖'
"""

query = load_product()
product = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

Running time of : 0 sec
 loading completed


In [5]:
product

,product
0,台股信用交易
1,結構型商品
2,債券型基金
3,雙向借券
4,海外債
5,不限用途
6,投資型保險商品
7,財管商品
8,基金定期定額
9,海外股票


In [6]:
def load_population1(this_prod,mother,ym,papulation_colname,papulation_train_value = 0):
    return f"""
select customer_id
    ,yyyymm
    ,NVL("{this_prod}{papulation_colname}",0) as {papulation_colname}
    ,NVL("{this_prod}Y",0) AS Y 
from S_IANLEONG.mlops_population a
where segment = '{mother}' and yyyymm = '{ym}' and NVL("{this_prod}{papulation_colname}",0) = {papulation_train_value}
"""
##例外(結構型)
def load_population2(this_prod,mother,ym,papulation_colname,papulation_train_value = 0):
    return f"""
select customer_id
    ,yyyymm
    ,NVL("{this_prod}{papulation_colname}",0) as {papulation_colname}
    ,NVL("{this_prod}Y",0) AS Y 
from S_IANLEONG.mlops_population a
where segment = '{mother}' and yyyymm = '{ym}' AND "98戶" is null and NVL("{this_prod}{papulation_colname}",0) = {papulation_train_value}
"""
##例外(客群上送，潛在高價值客戶)
def load_population3(this_prod,popu,ym,papulation_colname,papulation_train_value = 0):
    return f"""
select customer_id
    ,yyyymm
    ,NVL("{this_prod}{papulation_colname}R",0) as {papulation_colname}
    ,NVL("{this_prod}Y",0) AS Y 
from S_IANLEONG.mlops_population a
where yyyymm = '{ym}' and NVL("{this_prod}{papulation_colname}R",0) = {papulation_train_value}
"""
def load_population4(this_prod,mother,ym,papulation_colname,papulation_train_value = 0):
    return f"""
select customer_id
    ,yyyymm
    ,NVL("{this_prod}{papulation_colname}",0) as {papulation_colname}
    ,NVL("{this_prod}Y",1) AS Y 
from S_IANLEONG.mlops_population a
where yyyymm = '{ym}' and NVL("{this_prod}{papulation_colname}",0) = {papulation_train_value}
"""

def population_down_sampling(prd,popu,ym, market_flag_Y_N ):
    if prd in ['境內結構型','境外結構型','結構型商品']:
        papulation_colname = '近三年舊戶'
        query = load_population2(prd,popu,ym,papulation_colname)

        df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )
    elif prd in ['客群上送','潛在高價值客戶']:
        papulation_colname = '前季高交易量客戶'
        query = load_population3(prd,popu,ym,papulation_colname)
#         print(query)
        df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )
    elif prd in ['流失預警']:
        papulation_colname = '近一年實動'
        query = load_population4(prd,popu,ym,papulation_colname,papulation_train_value = 1)
#         print(query)
        df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )
    elif prd in ['海外股票']:
        papulation_colname = '近半年舊戶'
        query = load_population1(prd,popu,ym,papulation_colname)
#         print(query)
        df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )
    else :
        papulation_colname = '近一年舊戶'
        query = load_population1(prd,popu,ym,papulation_colname)
#         print(query)
        df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

    print(f'{ym}{prd}{popu}共{len(df)}筆')




    if market_flag_Y_N:
        limit_size = 150000
    else:
        limit_size = 400000

    if len(df) > limit_size:
        print(f'starting down sampling (year:{ym}) ...')
        df_y0 = df[df['y']==0]
        df_y1 = df[df['y']==1]
        df_y0_downsampled = resample(df_y0, random_state=42, n_samples=limit_size-len(df_y1), replace=False)
        #concat
        df_downsampled = pd.concat([df_y0_downsampled, df_y1])
        print(f'Successed!! down sampling (year:{ym})/ size:{len(df_downsampled)}) ...')
    else:
        print(f'don"t need down sampling (year:{ym}/ size:{len(df)}) ...')
        df_downsampled = df


    return df_downsampled


In [7]:
# prd = '潛在高價值客戶'
# popu = '不分潛客'
# ym = '202312'
# market_flag_Y_N = False
# # df = population_down_sampling(prd,popu,ym,market_flag_Y_N)

In [8]:
# papulation_colname = '前季高交易量客戶'
# query = load_population3(prd,popu,ym,papulation_colname,papulation_train_value = 0)
# print(query)
# df = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

In [9]:
# df

In [ ]:
pre_run_prod = []
pre_run_prod_finished = []

#mainly doing this thing
def load_sampling(prd,popu,ym):
    return f"""
select * from S_IANLEONG.mlops_population_sampling a
where product = '{prd}' and population = '{popu}' and yyyymm = '{ym}'
"""

for ap in product['product']:
    finished_status = True
    pre_run_prod.append(ap)

    if ap not in prod_ym_list_pd_12m:

        if ap not in ['客群上送','潛在高價值客戶']:
            market_flag_Y_N = True
            for m in mother_list1:#potential customers or not


                for ym in sorted(config.do_ym_list_pd_3m_detail)[0:-1]:

                    query = load_sampling(ap,m,ym)

                    sampling = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

                    if len(sampling)>0:
                        pass
                        print(f'{ym}{ap}{m}共{len(sampling)}筆')
                    else:
                        print(f'{ym}{ap}{m}無資料')
                        df = population_down_sampling(ap,m,ym,market_flag_Y_N)

                        df = df.assign(population = m)
                        df = df.assign(product = ap)
                        df = df[['customer_id','yyyymm','product','population']]

                        send_table_to_sql(targ = df,table_name='S_IANLEONG.mlops_population_sampling',account = config.account, pwd = config.pwd)
        else:# if ap in ['客群上送','潛在高價值客戶'] or else
            market_flag_Y_N = False
            for m in mother_list2:#do not distinguish potential customers or not
                for ym in sorted(config.do_ym_list_pd_3m)[0:-1]:
                    query = load_sampling(ap,m,ym)
                    sampling = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

                    if len(sampling)>0:
                        pass
                        print(f'{ym}{ap}{m}共{len(sampling)}筆')
                    else:
                        print(f'{ym}{ap}{m}無資料')
                        df = population_down_sampling(ap,m,ym,market_flag_Y_N)
                        df = df.assign(population = m)
                        df = df.assign(product = ap)
                        df = df[['customer_id','yyyymm','product','population']]
                        send_table_to_sql(targ = df,table_name='S_IANLEONG.mlops_population_sampling',account = config.account, pwd = config.pwd)
    else:#if ap in prod_ym_list_pd_12m or else
        market_flag_Y_N = False
        for m in mother_list2: #do not distinguish potential customers or not
            for ym in sorted(config.do_ym_list_pd_12m)[0:-1]:
                query = load_sampling(ap,m,ym)
                sampling = get_SQL_raw_data(query,account=config.account,pwd=config.pwd )

                if len(sampling)>0:
                    pass
                    print(f'{ym}{ap}{m}共{len(sampling)}筆')
                else:
                    df = population_down_sampling(ap,m,ym,market_flag_Y_N)
                    df = df.assign(population = m)
                    df = df.assign(product = ap)
                    df = df[['customer_id','yyyymm','product','population']]
                    send_table_to_sql(targ = df,table_name='S_IANLEONG.mlops_population_sampling',account = config.account, pwd = config.pwd)

Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202412台股信用交易潛客無資料
Running time of : 42 sec
 loading completed
202412台股信用交易潛客共2296974筆
starting down sampling (year:202412) ...
Successed!! down sampling (year:202412)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 3 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202501台股信用交易潛客無資料
Running time of : 40 sec
 loading completed
202501台股信用交易潛客共2304458筆
starting down sampling (year:202501) ...
Successed!! down sampling (year:202501)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202502台股信用交易潛客無資料
Running time of : 41 sec
 loading completed
202502台股信用交易潛客共2314863筆
starting down sampling (year:202502) ...
Successed!! down sampling (year:202502)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] R

Running time of : 42 sec
 loading completed
202504結構型商品潛客共2335407筆
starting down sampling (year:202504) ...
Successed!! down sampling (year:202504)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202505結構型商品潛客無資料
Running time of : 42 sec
 loading completed
202505結構型商品潛客共2337493筆
starting down sampling (year:202505) ...
Successed!! down sampling (year:202505)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202506結構型商品潛客無資料
Running time of : 44 sec
 loading completed
202506結構型商品潛客共2326590筆
starting down sampling (year:202506) ...
Successed!! down sampling (year:202506)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202

Successed!! down sampling (year:202508)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202509債券型基金潛客無資料
Running time of : 40 sec
 loading completed
202509債券型基金潛客共2314155筆
starting down sampling (year:202509) ...
Successed!! down sampling (year:202509)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202412債券型基金非潛客無資料
Running time of : 8 sec
 loading completed
202412債券型基金非潛客共378882筆
starting down sampling (year:202412) ...
Successed!! down sampling (year:202412)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202501債券型基金非潛客無資料
Running time of : 8 sec
 loading completed
202501債券型基金非潛客共379374筆
starting down sampling (yea

Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202503雙向借券非潛客無資料
Running time of : 8 sec
 loading completed
202503雙向借券非潛客共342189筆
starting down sampling (year:202503) ...
Successed!! down sampling (year:202503)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202504雙向借券非潛客無資料
Running time of : 8 sec
 loading completed
202504雙向借券非潛客共342283筆
starting down sampling (year:202504) ...
Successed!! down sampling (year:202504)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202505雙向借券非潛客無資料
Running time of : 8 sec
 loading completed
202505雙向借券非潛客共342833筆
starting down sampling (year:202505) ...
Successed!! down sampling (year:202505)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time 

use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202508海外債非潛客無資料
Running time of : 9 sec
 loading completed
202508海外債非潛客共411570筆
starting down sampling (year:202508) ...
Successed!! down sampling (year:202508)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202509海外債非潛客無資料
Running time of : 9 sec
 loading completed
202509海外債非潛客共412398筆
starting down sampling (year:202509) ...
Successed!! down sampling (year:202509)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202412不限用途潛客無資料
Running time of : 41 sec
 loading completed
202412不限用途潛客共2304493筆
starting down sampling (year:202412) ...
Successed!! down sampling (year:202412)/ size:150000) 

...

[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202501基金定期定額非潛客無資料
Running time of : 9 sec
 loading completed
202501基金定期定額非潛客共374461筆
starting down sampling (year:202501) ...
Successed!! down sampling (year:202501)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202502基金定期定額非潛客無資料
Running time of : 9 sec
 loading completed
202502基金定期定額非潛客共374160筆
starting down sampling (year:202502) ...
Successed!! down sampling (year:202502)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202503基金定期定額非潛客無資料
Running time of : 9 sec
 loading completed
202503基金定期定額非潛客共373822筆
starting down sampling (year:202503) ...
Successed!! down sampling (year:202503)/ size:150000) ...
use this: S_K

Running time of : 9 sec
 loading completed
202505海外股票非潛客共351235筆
starting down sampling (year:202505) ...
Successed!! down sampling (year:202505)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202506海外股票非潛客無資料
Running time of : 9 sec
 loading completed
202506海外股票非潛客共364745筆
starting down sampling (year:202506) ...
Successed!! down sampling (year:202506)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202507海外股票非潛客無資料
Running time of : 9 sec
 loading completed
202507海外股票非潛客共364100筆
starting down sampling (year:202507) ...
Successed!! down sampling (year:202507)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202508海外股

use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202412期貨潛客無資料
Running time of : 49 sec
 loading completed
202412期貨潛客共2299655筆
starting down sampling (year:202412) ...
Successed!! down sampling (year:202412)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202501期貨潛客無資料
Running time of : 50 sec
 loading completed
202501期貨潛客共2307060筆
starting down sampling (year:202501) ...
Successed!! down sampling (year:202501)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202502期貨潛客無資料
Running time of : 50 sec
 loading completed
202502期貨潛客共2317518筆
starting down sampling (year:202502) ...
Successed!! down sampling (year:202502)/ size:150000) ...
use 

Successed!! down sampling (year:202504)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202505基金潛客無資料
Running time of : 40 sec
 loading completed
202505基金潛客共2335115筆
starting down sampling (year:202505) ...
Successed!! down sampling (year:202505)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202506基金潛客無資料
Running time of : 41 sec
 loading completed
202506基金潛客共2324379筆
starting down sampling (year:202506) ...
Successed!! down sampling (year:202506)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202507基金潛客無資料
Running time of : 47 sec
 loading completed
202507基金潛客共2321798筆
starting down sampling (year:202507) ...
Succ

Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202509海外股票定期定額潛客無資料
Running time of : 46 sec
 loading completed
202509海外股票定期定額潛客共2259557筆
starting down sampling (year:202509) ...
Successed!! down sampling (year:202509)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202412海外股票定期定額非潛客無資料
Running time of : 9 sec
 loading completed
202412海外股票定期定額非潛客共368066筆
starting down sampling (year:202412) ...
Successed!! down sampling (year:202412)/ size:150000) ...
use this: S_KRISYJCHEN !Fubon003
[Success writing down to db] Running time : 2 sec
Running time of : 0 sec
 loading completed
length of dataframme is 0 !!
202501海外股票定期定額非潛客無資料
Running time of : 10 sec
 loading completed
202501海外股票定期定額非潛客共368307筆
starting down sampling (year:202501) ...
Successed!! down sampling (year:202501)/ size:150000) ...


In [22]:
!jupyter nbconvert --to script Retrain_new_mlops_double_preRun_popu.py

[NbConvertApp] WARNING | pattern 'Retrain_new_mlops_double_preRun_popu.py' matched no files
This application is used to convert notebook files (*.ipynb) to various other
formats.


Options
-------

Arguments that take values are actually convenience aliases to full
Configurables, whose aliases are listed on the help line. For more information
on full configurables, see '--help-all'.

--debug
    set log level to logging.DEBUG (maximize logging output)
--generate-config
    generate default config file
-y
    Answer yes to any questions instead of prompting.
--execute
    Execute the notebook prior to export.
--allow-errors
    Continue notebook execution even if one of the cells throws an error and include the error message in the cell output (the default behaviour is to abort conversion). This flag is only relevant if '--execute' was specified, too.
--stdin
    read a single notebook file from stdin. Write the resulting notebook with default basename 'notebook.*'
--stdout
    Write no